In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# TTA, ансамбли и потенциал специалистов

Здесь собраны сравнения моделей, TTA и ансамблей, а также **аудит потенциального** каскада специалистов. Специалисты не обучены: oracle/ceiling не является их измеренным качеством.

Начальные настройки PRIMARY/OTHER относятся к историческому сравнению ResNet50. В конце — завершённое сравнение с ResNet101. Обучение и test не запускаются. RUN_SUITE=False читает сохранённые результаты; True запускает только val inference и требует нового INFERENCE_RUN_ID.

Ошибки и неуверенность отдельной модели теперь анализируются в `05_error_analysis.ipynb`. Текущие выводы: `artifacts/reviews/resnet101_review.md`.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display

from archive.src.error_analysis import (
    load_predictions, model_comparison, class_metrics, confusion_pairs,
    plot_confusion, show_errors, show_train_examples, image_gallery,
    paired_comparison, specialist_routing_audit,
)
RAW_DIR = REPOSITORY_REPOSITORY_ROOT / "data" / "raw"


In [ ]:
EXPERIMENTS = [
    "resnet50_partial_control_001", "resnet50_full_001",
    "resnet50_partial_domain_shift_001",
    "resnet50_partial_domain_shift_stronger_001",
]
PRIMARY = "resnet50_partial_domain_shift_stronger_001"
OTHER = "resnet50_partial_domain_shift_001"
if PRIMARY not in EXPERIMENTS or OTHER not in EXPERIMENTS:
    raise ValueError("PRIMARY и OTHER должны входить в EXPERIMENTS")
frame, classes = load_predictions(PROJECT_ROOT, EXPERIMENTS)
names = classes.set_index("label").class_name.to_dict()
display(model_comparison(frame, EXPERIMENTS).round(3))
print(f"Проверены {len(frame)} уникальных val IDs и raw ground truth.")


## Взаимодополнение моделей

Oracle использует известную разметку и **не является ансамблем**. Реальные soft ensembles ниже используют сохранённые полные probabilities.


In [ ]:
display(paired_comparison(frame, PRIMARY, OTHER).to_frame("value").round(3))
primary_ok = frame[PRIMARY].eq(frame.true_label)
other_ok = frame[OTHER].eq(frame.true_label)
for mask, title in [
    (~primary_ok & other_ok, "Other correct; primary wrong"),
    (primary_ok & ~other_ok, "Primary correct; other wrong"),
]:
    rows = [
        (image_id, f"True: {names[row.true_label]}\nPrimary: {names[row[PRIMARY]]}\nOther: {names[row[OTHER]]}")
        for image_id, row in frame.loc[mask].iterrows()
    ]
    image_gallery(rows, RAW_DIR / "val", title)


## Потенциал и ограничения каскада специалистов

Маршрут определяет **предсказанный** класс. Ground truth используется только для аудита. Оцениваем ошибки внутри группы, правильные ответы под риском изменения, посторонние снимки (нужен reject/other), ошибки, не доходящие до специалиста.

Church → medium_residential не попадёт специалисту church/palace/commercial_area. Поэтому жёсткая маршрутизация по argmax может упустить нужные снимки; после появления probabilities полезнее мягкое объединение или маршрут по top-2.

In [ ]:
GROUPS = {
    "buildings": ["church", "palace", "commercial_area"],
    "water": ["lake", "wetland"],
    "courts": ["basketball_court", "tennis_court"],
}
routing = specialist_routing_audit(frame, PRIMARY, classes, GROUPS)
display(routing)
fixable = int(routing.within_group_errors_potentially_fixable.sum())
current_correct = int(primary_ok.sum())
print(f"Внутри групп потенциально исправимы {fixable} / {int((~primary_ok).sum())} ошибок.")
print(f"Идеальный ceiling без ухудшений: {(current_correct + fixable) / len(frame):.2%}.")
print("Это НЕ качество реальных специалистов: они тоже ошибаются и могут портить правильные ответы.")


## Реальные TTA и ансамбль сохранённых checkpoints

Исторический quality_inference_001 уже выполнен. По умолчанию читаем его результаты. Для нового сравнения с ResNet101 включите её checkpoint, выберите новый ID и RUN_SUITE=True. Suite вычисляет none/flips/D4 и равновесные ансамбли, разделяя forwards. Обучение и test не запускаются; experiments.csv не меняется.


In [ ]:
import json
import pandas as pd
from archive.src.inference import run_validation_suite
from archive.src.training.predict import load_probabilities

CHECKPOINTS = {
    "stronger": "artifacts/models/resnet50_partial_domain_shift_stronger_001/best.pt",
    "mild": "artifacts/models/resnet50_partial_domain_shift_001/best.pt",
    # "resnet101": "artifacts/models/resnet101_partial_domain_shift_stronger_001/best.pt",
}
INFERENCE_RUN_ID = "quality_inference_001"
RUN_SUITE = False  # True после добавления новой модели; задайте НОВЫЙ ID
if RUN_SUITE:
    run_validation_suite(CHECKPOINTS, INFERENCE_RUN_ID, project_root=PROJECT_ROOT,
                         device="auto", raw_dir=RAW_DIR, batch_size=16,
                         tta_modes=("none", "flips", "d4"), cache_from="quality_inference_001")
INFERENCE_LOGS = PROJECT_ROOT / "artifacts/logs" / INFERENCE_RUN_ID
INFERENCE_PREDS = PROJECT_ROOT / "artifacts/predictions" / INFERENCE_RUN_ID
if not (INFERENCE_LOGS / "selection.json").exists():
    raise FileNotFoundError("Сначала выполните suite с доступными checkpoints")
comparison = pd.read_csv(INFERENCE_LOGS / "comparison.csv")
selection = json.loads((INFERENCE_LOGS / "selection.json").read_text(encoding="utf-8"))
display(comparison)
print("Выбранный рецепт:", selection["recipe"])


In [ ]:
selected_probs = load_probabilities(INFERENCE_PREDS / f"{selection['recipe']}.npz")
scored = selected_probs.table(include_scores=True)
analysis_frame = scored.set_index("id").rename(columns={"label": "selected_prediction"})
display(class_metrics(analysis_frame, "selected_prediction", classes).round(2))
display(confusion_pairs(analysis_frame, "selected_prediction", classes))
plot_confusion(analysis_frame, "selected_prediction", classes)
wrong = scored.loc[scored.label.ne(scored.true_label)]
display(wrong.sort_values("confidence", ascending=False))
image_gallery([
    (r.id, f"True: {names[r.true_label]}\nPred: {names[r.label]}\nConfidence: {r.confidence:.2f}, margin: {r.margin:.2f}")
    for r in wrong.itertuples()
], RAW_DIR / "val", "Errors of selected inference recipe")


In [ ]:
baseline = load_probabilities(INFERENCE_PREDS / "stronger__none.npz").table().set_index("id")
selected = scored.set_index("id").loc[baseline.index]
baseline_ok = baseline.label.eq(baseline.true_label)
selected_ok = selected.label.eq(selected.true_label)
print("Исправлено ошибок stronger:", int((~baseline_ok & selected_ok).sum()))
print("Добавлено новых ошибок:", int((baseline_ok & ~selected_ok).sum()))
display(selected.loc[baseline_ok & ~selected_ok])


Рецепт в selection.json выбран по val Accuracy, затем по NLL и числу views при совпадении метрик. Это наблюдаемое качество на той же val; уверенность softmax не является доказательством правильности разметки. 99_final_pipeline.ipynb воспроизводит фиксированный manifest; test inference там включается отдельно.


## ResNet101: завершённый запуск и сравнение сохранённых probabilities
Исходный запуск: `resnet101_partial_domain_shift_stronger_001`. Ниже нет обучения, новых forward-pass или чтения test. ResNet101 использует **none**, ResNet50 — ранее рассчитанные TTA. См. `artifacts/reviews/resnet101_review.md`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

review_root = PROJECT_ROOT
review = json.loads((review_root / "artifacts/reviews/resnet101_review_audit.json").read_text(encoding="utf-8"))
display(pd.DataFrame(review["comparison"]).drop(columns="members"))
display(pd.DataFrame(review["class_metrics"]["resnet101_none"]).query("errors > 0").sort_values("errors", ascending=False))
display(pd.DataFrame(review["resnet101_errors"]))
display(Image(filename=str(review_root / "artifacts/reviews/resnet101_errors.png")))


## ResNet101: global/local diagnostic и фиксированное объединение

Полный кадр256, none/flips/D4 и пять crops160 (углы+центр), каждый resize→256. Все вероятности сохраняются. Никакого нового обучения или чтения test. Локальный запуск `resnet101_global_local_001` ниже используется как готовый кэш.

На Kaggle: укажите PROJECT_ROOT и RAW_DIR, положите обученный best.pt в CHECKPOINT и задайте новый LOCAL_RUN_ID; RUN_LOCAL=True. BATCH_SIZE можно увеличить на GPU. RESUME_LOCAL=True возобновляет прерванный запуск с тех же батчей; изменять batch_size при resume нельзя. Завершённый кэш только читается.


In [ ]:
from archive.src.local_inference import run_local_diagnostic, analyze_saved_local
from archive.src.local_analysis import show_local_cases

LOCAL_RUN_ID = "resnet101_global_local_001"
CHECKPOINT = "artifacts/models/resnet101_partial_domain_shift_stronger_001/best.pt"
RUN_LOCAL = False
RESUME_LOCAL = True
LOCAL_BATCH_SIZE = 4  # CPU4; для нового GPU-run можно16
LOCAL_CPU_THREADS = 4
if RUN_LOCAL:
    run_local_diagnostic(CHECKPOINT, LOCAL_RUN_ID, project_root=PROJECT_ROOT, raw_dir=RAW_DIR,
                         device="auto", batch_size=LOCAL_BATCH_SIZE, cpu_threads=LOCAL_CPU_THREADS,
                         crop_size=160, num_workers=0, resume=RESUME_LOCAL)
local_review = analyze_saved_local(LOCAL_RUN_ID, project_root=PROJECT_ROOT, raw_dir=RAW_DIR)
local_logs = local_review["paths"]["logs"]
local_preds = local_review["paths"]["predictions"]
display(local_review["comparison"])
print(json.dumps(local_review["summary"], ensure_ascii=False, indent=2))
print("Текущий лучший рецепт:", local_review["selection"]["recipe"])


### Где crops показывают правильный объект

Oracle any-crop top1 использует true labels только для диагностики. **Не является качеством ансамбля.** Top3 отдельно отмечает только новые случаи, которых ещё не было в full top3. Зелёный заголовок — правильный ответ view, красный — ошибочный.


In [ ]:
diagnostic = local_review["diagnostic"]
crop_recovered = diagnostic.loc[~diagnostic.full_correct & diagnostic.any_crop_true_top1]
display(crop_recovered)
MAX_LOCAL_CASES = 12  # None: все; влияет только на отображение
show_local_cases(crop_recovered.id.head(MAX_LOCAL_CASES), RAW_DIR, local_preds, classes,
                 crop_size=160, page_size=3)
plt.close("all")


### Два работающих правила fusion: оцениваем исправления и регрессии

75% full +25% local: mean пяти crop-распределений либо поэлементный max с нормировкой по классам. Веса фиксированы до расчёта; ничего не подбираем по отдельным ошибкам. Сравнение всей val, включая уже правильные ответы, важнее oracle.


In [ ]:
for method in ["mean", "max"]:
    prediction = load_probabilities(local_preds / f"global_local_{method}.npz")
    baseline = load_probabilities(local_preds / "none.npz").table().set_index("id")
    candidate = prediction.table(include_scores=True).set_index("id").loc[baseline.index]
    baseline_ok = baseline.label.eq(baseline.true_label)
    candidate_ok = candidate.label.eq(candidate.true_label)
    print(method, "исправлено:", int((~baseline_ok & candidate_ok).sum()),
          "добавлено ошибок:", int((baseline_ok & ~candidate_ok).sum()))
    display(candidate.loc[baseline_ok & ~candidate_ok])
    display(class_metrics(candidate.rename(columns={"label": "prediction"}), "prediction", classes).round(2))


`final_selection.json` содержит выбранное работающее правило. Обычные ориентации/ансамбли — format_version1; global/local — format_version2. `run_selected_inference` поддерживает оба, включая явный test inference после окончательного выбора. Этот ноутбук test не запускает.

Следующее обучение определяется фактическим crop recovery и регрессиями. См. `artifacts/reviews/quality_experiment_plan.md` и отчёт `artifacts/reviews/global_local_review.md`, подготовленный после завершения локального расчёта.
